In [1]:
# ==========================================
# 0. INSTALL DEPENDENCIES
# ==========================================
!pip install -q gdown datasets sqlparse

import json
import os
import gdown
import zipfile
import re
import random
import sqlparse
from datasets import load_dataset
from tqdm import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
OUTPUT_FILE = "nl2sql_finetuning_dataset.json"

# ✅ CRITICAL CHANGE 1: Set to 0 to prevent "WikiSQL Poisoning"
# We only want high-quality, complex logic examples now.
WIKISQL_CAP = 0 

SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL model. Your goal is to generate a valid SQLite query "
    "to answer the user's question, using ONLY the tables, columns, and relationships "
    "shown in the provided schema. "
    "Do NOT invent identifiers (tables or columns) that do not exist. "
    "Output only the SQL query code, without any markdown formatting or explanation."
)

# Regex to catch "garbage" column names common in WikiSQL (just in case we re-enable it later)
BAD_COLUMN_PATTERN = re.compile(r'\b(col\d+|column\s*\d+|header|untitled|field\d+)\b', re.IGNORECASE)

# ==========================================
# 2. GOOGLE DRIVE DOWNLOADER
# ==========================================
def download_and_extract_drive(name, file_id):
    """Downloads from Google Drive using gdown and extracts."""
    if os.path.exists(name) or os.path.exists(f"{name}_dataset"):
        print(f"✅ {name} already exists.")
        return

    url = f'https://drive.google.com/uc?id={file_id}'
    output = f"{name}.zip"
    
    print(f"⬇️ Downloading {name}...")
    gdown.download(url, output, quiet=False)
    
    print(f"📦 Extracting {name}...")
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall(".")
    
    # Cleanup
    os.remove(output)
    print(f"✅ Ready: {name}")

# ==========================================
# 3. ROBUST SCHEMA PARSER (With Casing Fixes & Type Preservation)
# ==========================================
def get_schema_string(table_json_path, db_id, preserve_column_types=True):
    """Parses standard Spider 1.0 format (used by Spider, SParC, CoSQL).
    
    NOW INCLUDES:
    - Preserved column types (instead of always TEXT)
    - Original table/column mapping for query normalization
    """
    try:
        with open(table_json_path, 'r') as f:
            tables = json.load(f)
        
        schema_data = next((t for t in tables if t['db_id'] == db_id), None)
        if not schema_data: return "", {}

        table_names = schema_data['table_names_original']
        column_names = schema_data['column_names_original']
        primary_keys = schema_data.get('primary_keys', [])
        foreign_keys = schema_data.get('foreign_keys', [])
        column_types = schema_data.get('column_types', ['text'] * len(column_names))
        
        # ✅ FIX 1: Build mapping for query normalization (original → lowercase)
        col_mapping = {}  # Maps original names to lowercase
        for col_idx, (t_id, col_name) in enumerate(column_names):
            if t_id < 0: continue
            col_mapping[col_name] = col_name.lower()
        
        table_mapping = {}
        for t_id, t_name in enumerate(table_names):
            table_mapping[t_name] = t_name.lower()
        
        table_cols = {}
        for col_idx, (t_id, col_name) in enumerate(column_names):
            if t_id < 0: continue
            
            safe_col_name = col_name.lower()
            
            # ✅ FIX 2: Preserve column types instead of forcing TEXT
            col_type = "TEXT"
            if preserve_column_types and col_idx < len(column_types):
                col_type_str = column_types[col_idx].upper()
                # Map common types
                if 'int' in col_type_str.lower():
                    col_type = "INT"
                elif 'real' in col_type_str.lower() or 'float' in col_type_str.lower():
                    col_type = "REAL"
                elif 'date' in col_type_str.lower():
                    col_type = "DATE"
                elif 'time' in col_type_str.lower():
                    col_type = "DATETIME"
                elif 'bool' in col_type_str.lower():
                    col_type = "BOOLEAN"
                else:
                    col_type = "TEXT"
            
            table_cols.setdefault(t_id, []).append((col_idx, safe_col_name, col_type))

        create_stmts = []
        for t_id, t_name in enumerate(table_names):
            safe_t_name = t_name.lower()
            
            cols = table_cols.get(t_id, [])
            col_defs = []
            for c_idx, c_name, c_type in cols:
                col_def = f"{c_name} {c_type}"
                if c_idx in primary_keys: col_def += " PRIMARY KEY"
                col_defs.append(col_def)
            
            # Foreign keys (Normalized to lowercase)
            for src, tgt in foreign_keys:
                if src in [c[0] for c in cols]:
                    tgt_col_info = column_names[tgt]
                    tgt_t_name = table_names[tgt_col_info[0]].lower() 
                    tgt_c_name = tgt_col_info[1].lower()              
                    src_c_name = column_names[src][1].lower()         
                    col_defs.append(f"FOREIGN KEY ({src_c_name}) REFERENCES {tgt_t_name}({tgt_c_name})")

            stmt = f"CREATE TABLE {safe_t_name} (\n  " + ",\n  ".join(col_defs) + "\n);"
            create_stmts.append(stmt)
            
        return "\n\n".join(create_stmts), {**col_mapping, **table_mapping}
    except Exception as e:
        print(f"Error parsing schema: {e}")
        return "", {}

def clean_sql(sql):
    return sql.replace('"', "'").strip() if sql else ""

def normalize_query_casing(query, identifier_mapping):
    """✅ FIX 1: Normalize query to match schema casing.
    
    Replaces original table/column names with lowercase equivalents.
    """
    if not query or not identifier_mapping:
        return clean_sql(query)
    
    normalized = query
    # Sort by length (longest first) to avoid partial replacements
    for original, lowercase in sorted(identifier_mapping.items(), key=lambda x: -len(x[0])):
        # Use word boundaries to match whole identifiers
        pattern = r'\b' + re.escape(original) + r'\b'
        normalized = re.sub(pattern, lowercase, normalized, flags=re.IGNORECASE)
    
    return clean_sql(normalized)

def validate_sql(query, schema_text):
    """✅ FIX 3: Validate SQL query for basic correctness.
    
    Checks:
    1. Query is not empty
    2. Query contains expected keywords (SELECT, FROM, etc.)
    3. Query doesn't reference non-existent tables/columns
    """
    query_clean = query.strip()
    
    # Check 1: Not empty
    if not query_clean:
        return False, "Empty query"
    
    # Check 2: Contains SELECT (basic SQL check)
    if 'SELECT' not in query_clean.upper():
        return False, "No SELECT statement"
    
    # Check 3: Extract table names from schema
    schema_lower = schema_text.lower()
    create_table_matches = re.findall(r'CREATE TABLE (\w+)', schema_lower)
    valid_tables = set(create_table_matches)
    
    # Check 4: Extract referenced tables in query
    try:
        from_matches = re.findall(r'\bFROM\s+(\w+)', query_clean, re.IGNORECASE)
        join_matches = re.findall(r'\bJOIN\s+(\w+)', query_clean, re.IGNORECASE)
        referenced_tables = set(t.lower() for t in from_matches + join_matches)
        
        # All referenced tables must exist
        if referenced_tables and not referenced_tables.issubset(valid_tables):
            return False, f"References unknown tables: {referenced_tables - valid_tables}"
    except:
        pass
    
    # Check 5: Avoid known problem patterns
    if re.search(r'SELECT\s+\*\s+.*GROUP\s+BY', query_clean, re.IGNORECASE):
        return False, "SELECT * with GROUP BY (may be invalid)"
    
    return True, "Valid"

# ==========================================
# 4. DATA PROCESSORS
# ==========================================
def process_spider():
    formatted = []
    print("Processing Spider 1.0 (Train + Others)...")
    
    # ✅ CRITICAL CHANGE 3: Include 'train_others.json'
    # Adds ~1,650 extra complex examples to replace the lost WikiSQL data
    files_to_process = ["spider/train_spider.json", "spider/train_others.json"]
    
    invalid_count = 0
    for file_path in files_to_process:
        if not os.path.exists(file_path): 
            print(f"⚠️ Skipping {file_path} (not found)")
            continue
            
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        for item in tqdm(data, desc=f"Loading {file_path}"):
            schema, id_mapping = get_schema_string("spider/tables.json", item['db_id'])
            if not schema: continue
            
            # ✅ FIX 1: Normalize query casing to match schema
            normalized_query = normalize_query_casing(item['query'], id_mapping)
            
            # ✅ FIX 3: Validate query before adding
            is_valid, reason = validate_sql(normalized_query, schema)
            if not is_valid:
                invalid_count += 1
                continue
            
            formatted.append({
                "conversations": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion:\n{item['question']}"},
                    {"role": "assistant", "content": normalized_query}
                ]
            })
    
    if invalid_count > 0:
        print(f"⚠️ Skipped {invalid_count} invalid Spider queries")
    return formatted

def process_conversational(folder_name, dataset_type):
    formatted = []
    print(f"Processing {dataset_type} (Multi-Turn Cumulative - Only Final State)...")
    
    if dataset_type == "CoSQL":
        data_path = f"{folder_name}/sql_state_tracking/cosql_train.json"
    else: # SParC
        data_path = f"{folder_name}/train.json"
    
    tables_path = f"{folder_name}/tables.json"

    if not os.path.exists(data_path): 
        print(f"⚠️ Path not found: {data_path}")
        return []

    with open(data_path, 'r') as f:
        data = json.load(f)
    
    invalid_count = 0
    for item in tqdm(data):
        schema, id_mapping = get_schema_string(tables_path, item['database_id'])
        if not schema: continue

        # Initialize history with System Prompt
        current_history = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]
        
        for idx, turn in enumerate(item['interaction']):
            # First turn gets schema
            if idx == 0:
                user_content = f"Schema:\n{schema}\n\nQuestion:\n{turn['utterance']}"
            else:
                user_content = turn['utterance']

            current_history.append({"role": "user", "content": user_content})
            
            # ✅ FIX 1: Normalize query casing to match schema
            normalized_query = normalize_query_casing(turn['query'], id_mapping)
            
            # ✅ FIX 3: Validate query before adding
            is_valid, reason = validate_sql(normalized_query, schema)
            if not is_valid:
                invalid_count += 1
                # Still add to history for context, but mark as invalid
                current_history.append({"role": "assistant", "content": normalized_query})
                continue
            
            current_history.append({"role": "assistant", "content": normalized_query})
        
        # ✅ FIX 4: Only save FINAL cumulative state (not every intermediate turn)
        # This prevents near-duplicate training examples
        if len(current_history) > 1:  # Has at least system + one turn
            formatted.append({
                "conversations": list(current_history)
            })
    
    if invalid_count > 0:
        print(f"⚠️ Skipped {invalid_count} invalid {dataset_type} queries")
    return formatted

def process_wikisql():
    # ✅ Early Exit for WikiSQL
    if WIKISQL_CAP == 0:
        print("🚫 WikiSQL Disabled (Correct choice for high-performance tuning).")
        return []

    formatted = []
    print(f"Processing WikiSQL (Capped at {WIKISQL_CAP})...")
    ds = load_dataset("b-mc2/sql-create-context", split="train")
    
    ds = ds.shuffle(seed=42)
    count = 0
    skipped_garbage = 0
    
    for row in tqdm(ds):
        if count >= WIKISQL_CAP: break
        
        schema = row['context']
        sql = row['answer']
        question = row['question']

        # Filters
        if "(Id VARCHAR)" in schema or schema.count(",") < 1 or BAD_COLUMN_PATTERN.search(schema):
            skipped_garbage += 1
            continue

        formatted.append({
            "conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Schema:\n{schema}\n\nQuestion:\n{question}"},
                {"role": "assistant", "content": clean_sql(sql)}
            ]
        })
        count += 1
    
    print(f"🚫 Skipped {skipped_garbage} garbage rows.")
    return formatted

def process_gretel(num_samples=10000):
    """
    Loads gretelai/synthetic_text_to_sql, subsamples it, and formats it
    to match our conversational schema.
    """
    print(f"\n🚀 Processing GretelAI Synthetic Dataset (Subset: {num_samples})...")
    try:
        from datasets import load_dataset
        # Load dataset from Hugging Face
        dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")
        
        # Shuffle and Select Subset
        if len(dataset) > num_samples:
            dataset = dataset.shuffle(seed=42).select(range(num_samples))
        
        formatted_data = []
        
        for item in tqdm(dataset, desc="Formatting Gretel Data"):
            # Gretel structure: {'sql_prompt': ..., 'sql_context': ..., 'sql': ...}
            # We need to parse 'sql_context' to get the schema if possible, 
            # OR just use the provided context as the schema string directly.
            
            schema_str = item.get('sql_context', '')
            question = item.get('sql_prompt', '')
            sql = item.get('sql', '')
            
            if not schema_str or not question or not sql:
                continue

            # Validate SQL (basic check)
            if not sql.strip().upper().startswith("SELECT"):
                continue

            # Format in our conversational structure
            # System Prompt -> User (Schema + Question) -> Assistant (SQL)
            
            system_msg = {
                "role": "system",
                "content": SYSTEM_PROMPT
            }
            
            user_content = f"Schema:\n{schema_str}\n\nQuestion:\n{question}"
            
            user_msg = {
                "role": "user",
                "content": user_content
            }
            
            assistant_msg = {
                "role": "assistant",
                "content": sql
            }
            
            formatted_data.append({
                "conversations": [system_msg, user_msg, assistant_msg]
            })
            
        print(f"✅ Loaded {len(formatted_data)} examples from GretelAI.")
        return formatted_data

    except Exception as e:
        print(f"❌ Error processing Gretel dataset: {e}")
        return []

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Download from Drive
    download_and_extract_drive("spider", "1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J")
    download_and_extract_drive("sparc", "1Uu7NMHTR1tdQw1t7bAuM7OPU4LElVKfg")
    download_and_extract_drive("cosql_dataset", "1Y3ydpFiQQ3FC0bzdfy3groV95O_f1nXF")
    
    final_data = []
    
    # 2. Process
    final_data.extend(process_spider())
    final_data.extend(process_conversational("sparc", "SParC"))
    final_data.extend(process_conversational("cosql_dataset", "CoSQL"))
    # ✅ Include Augmented Data
    final_data.extend(process_gretel(num_samples=10000))
    final_data.extend(process_wikisql())
    
    # 3. Save
    print(f"\n📊 Final Dataset Count: {len(final_data)}")
    with open(OUTPUT_FILE, "w") as f:
        json.dump(final_data, f, indent=2)
    print(f"✅ Saved to {OUTPUT_FILE}")

    # 4. Zip for easy download
    !zip -r sql_dataset.zip nl2sql_finetuning_dataset.json
    from IPython.display import FileLink
    display(FileLink(r'sql_dataset.zip'))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 36.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 23.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 23.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
⬇️ Downloading spider...


Downloading...
From (original): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J
From (redirected): https://drive.google.com/uc?id=1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J&confirm=t&uuid=ea825b1e-0dca-488c-9abe-ccc6a7be99dc
To: /kaggle/working/spider.zip
100%|██████████| 206M/206M [00:04<00:00, 50.4MB/s]


📦 Extracting spider...
✅ Ready: spider
⬇️ Downloading sparc...


Downloading...
From (original): https://drive.google.com/uc?id=1Uu7NMHTR1tdQw1t7bAuM7OPU4LElVKfg
From (redirected): https://drive.google.com/uc?id=1Uu7NMHTR1tdQw1t7bAuM7OPU4LElVKfg&confirm=t&uuid=38a4a312-7d1b-47b8-848b-cd12944192bd
To: /kaggle/working/sparc.zip
100%|██████████| 99.4M/99.4M [00:01<00:00, 77.4MB/s]


📦 Extracting sparc...
✅ Ready: sparc
⬇️ Downloading cosql_dataset...


...

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

Formatting Gretel Data: 100%|██████████| 10000/10000 [00:01<00:00, 7055.26it/s]


✅ Loaded 8915 examples from GretelAI.
🚫 WikiSQL Disabled (Correct choice for high-performance tuning).

📊 Final Dataset Count: 14104
✅ Saved to nl2sql_finetuning_dataset.json
  adding: nl2sql_finetuning_dataset.json (deflated 90%)


/kaggle/working/sql_dataset.zip